> **수정본 안내 (2026-10 우병오)** — 현재 Colab에서 실행되도록 수정한 버전이다.
> - **BERT 학습 코드 교체**: 원본의 `%tensorflow_version 1.x` + `bert-tensorflow` + TF Hub(`hub.Module`)는 현재 Colab에서 동작하지 않는다 → 같은 BERT-Base(Cased) 모델을 Hugging Face `transformers`(PyTorch)로 미세조정하도록 변경 (배치 16, 3 에폭, 학습률 2e-5, 웜업 10%, 최대 길이 128 — 원본 설정 유지)
> - **버그 수정**: `load_data`에서 `data.sample(frac=1)` 결과를 저장하지 않아 데이터가 섞이지 않던 문제 수정
> - **데이터 주소 수정**: SST-2(만료된 firebase 주소 → GLUE 미러), MRPC(msi 수동 업로드 → SentEval 텍스트 파일 자동 다운로드), 공손함 데이터 미러 추가
> - **런타임**: GPU(T4) 권장


## Dataset

데이터 파일을 불러와서 pandas DataFrame으로 구성하는 방법을 실습해보겠습니다.

우선 학습 데이터를 다운로드하고 압축을 풉니다.

데이터 출처: http://www.cs.cornell.edu/~cristian/Politeness.html

In [ ]:
import os, urllib.request
# 원본 주소(코넬대)가 응답하지 않을 때는 GitHub 미러에서 받는다.
URLS = ["http://www.cs.cornell.edu/~cristian/Politeness_files/Stanford_politeness_corpus.zip",
        "https://github.com/meyersbs/politeness/raw/master/politeness/data/Stanford_politeness_corpus.zip"]
if not os.path.exists("Stanford_politeness_corpus.zip"):
  for url in URLS:
    try:
      urllib.request.urlretrieve(url, "Stanford_politeness_corpus.zip"); print("downloaded:", url); break
    except Exception as e:
      print("실패:", url, e)

if not os.path.exists("Stanford_politeness_corpus/wikipedia.annotated.csv"):
  !unzip -q -o Stanford_politeness_corpus.zip


다운로드된 데이터셋은 CSV 파일 형태로 되어 있습니다.

파일이 어떻게 구성되어 있는지 살펴볼까요?

In [ ]:
!head Stanford_politeness_corpus/wikipedia.annotated.csv

한 줄이 하나의 데이터를 나타내고, 각 데이터 값들은 쉼표로 구분되어 있는 것을 볼 수 있습니다.

또한, 첫 줄은 각 데이터 값들이 어떤 항목을 나타내는지 알려주는 헤더입니다.

이런 형식의 데이터 파일은 pandas의 read_csv 함수를 이용하여 쉽게 불러올 수 있습니다.

참고: [pandas.read_csv](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html)

In [ ]:
import urllib
import pandas as pd

data = pd.read_csv("Stanford_politeness_corpus/wikipedia.annotated.csv")
pd.set_option('display.max_columns', None)

print(data.head())

이번엔 다른 형태의 데이터를 다운로드 받아보겠습니다. [GLUE dataset](https://gluebenchmark.com) 중 하나인 Stanford Sentiment Treebank(SST-2) 입니다.

[수정] 원본의 firebase 다운로드 주소가 만료되어 GLUE 공식 미러(dl.fbaipublicfiles.com) 주소로 바꿨습니다.

In [ ]:
import os
import urllib.request

download_url = "https://dl.fbaipublicfiles.com/glue/data/SST-2.zip"

if not os.path.exists("sst.zip"):
  urllib.request.urlretrieve(download_url, "sst.zip")

if not os.path.exists("SST-2/train.tsv"):
  !unzip -q -o sst.zip

!head SST-2/train.tsv


이번에도 한 줄이 하나의 데이터를 나타냅니다. 

하지만 이번에는 데이터 항목들 간의 구분자가 쉼표가 아닌 탭입니다.

pandas.read_csv 함수는 sep 파라미터를 이용하여 구분자를 지정해줄 수 있습니다. 앞선 사용 예에서는 sep이 지정되지 않았는데, 이 때는 쉼표가 기본값으로 사용됩니다.

다시 한 번 pandas.read_csv 함수를 이용하여 이 데이터셋을 불러와보겠습니다.

In [ ]:
data = pd.read_csv("SST-2/train.tsv", sep='\t')
pd.set_option('display.max_columns', None)

print(data.head())

파일 파싱 에러(ParseError) 등으로 인해 read_csv 함수를 사용할 수 없는 경우도 있습니다.

그런 경우는 우선 python을 이용하여 데이터를 배열 형태로 불러오고, 이로부터 pandas DataFrame을 생성할 수 있습니다.

Microsoft Research Paraphrase Corpus(MRPC)를 이용하여 이러한 경우를 실습해보겠습니다.

[수정] 원본은 마이크로소프트 홈페이지에서 msi 파일을 직접 받아 업로드해야 했지만, 같은 데이터를 텍스트 파일로 제공하는 SentEval 미러에서 바로 내려받도록 바꿨습니다.

In [ ]:
import os

os.makedirs("MRPC", exist_ok=True)
for split in ["train", "test"]:
  path = f"MRPC/msr_paraphrase_{split}.txt"
  if not os.path.exists(path):
    urllib.request.urlretrieve(f"https://dl.fbaipublicfiles.com/senteval/senteval_data/msr_paraphrase_{split}.txt", path)

!head -n 5 MRPC/msr_paraphrase_train.txt


다음으로 python으로 파일을 읽고 각 줄을 탭을 기준으로 분리하는 과정을 통해 DataFrame을 생성해보겠습니다.

이 경우에는 columns 파라미터를 통해 헤더를 직접 지정해줘야 합니다.

참고: [pandas.DataFrame](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html)

In [ ]:
with open("MRPC/msr_paraphrase_train.txt", 'r', encoding='utf-8-sig') as fh:
  raw_data = [line.strip().split('\t') for line in fh]

data = pd.DataFrame(raw_data[1:], columns=raw_data[0])

print(data.head())


## BERT를 이용한 Text Classification

BERT를 이용한 text classification 모델을 만들고 학습시켜보겠습니다.

[수정] 원본은 TensorFlow 1.x 전용인 `bert-tensorflow` 패키지와 TF Hub 모듈(`hub.Module`)을 사용했지만, 현재 Colab은 TensorFlow 1.x를 지원하지 않아 실행되지 않습니다.
같은 사전학습 모델(**BERT-Base, Cased**: 12층, 768차원, 12헤드)을 **Hugging Face `transformers`**(PyTorch)로 불러와 동일한 방식(사전학습된 BERT + 분류층 1개를 함께 미세조정)으로 학습합니다.

⚙️ **런타임 → 런타임 유형 변경 → GPU(T4)** 를 선택하고 실행하세요. (CPU에서는 매우 느립니다.)

참고: [Hugging Face BERT](https://huggingface.co/google-bert/bert-base-cased), [BERT github](https://github.com/google-research/bert)

In [ ]:
# Colab에는 transformers가 기본 설치되어 있다. (없으면 설치)
!pip install -q transformers

In [ ]:
import sys
sys.modules.setdefault("tensorflow", None)   # TensorBoard 기록 시 TensorFlow를 불러오지 않게 함 (PyTorch와 함께 로드될 때의 충돌 방지)
import math
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


def make_features(dataset, tokenizer, max_seq_length, data_column, label_column):
    """문장 → BERT 입력(input_ids, attention_mask, token_type_ids) 변환"""
    enc = tokenizer(dataset[data_column].tolist(), padding="max_length", truncation=True,
                    max_length=max_seq_length, return_tensors="pt")
    labels = torch.tensor(dataset[label_column].tolist(), dtype=torch.long)
    return TensorDataset(enc["input_ids"], enc["attention_mask"], enc["token_type_ids"], labels)


def evaluate(model, loader):
    """테스트 데이터에 대한 손실과 정확도·F1·AUC 등 평가 지표 계산"""
    model.eval()
    losses, probs, preds, golds = [], [], [], []
    with torch.no_grad():
        for input_ids, attn, seg, labels in loader:
            out = model(input_ids=input_ids.to(device), attention_mask=attn.to(device),
                        token_type_ids=seg.to(device), labels=labels.to(device))
            losses.append(out.loss.item())
            p = torch.softmax(out.logits, dim=-1)[:, 1]
            probs += p.cpu().tolist()
            preds += out.logits.argmax(-1).cpu().tolist()
            golds += labels.tolist()
    tn, fp, fn, tp = confusion_matrix(golds, preds, labels=[0, 1]).ravel()
    return {"loss": float(np.mean(losses)),
            "eval_accuracy": accuracy_score(golds, preds),
            "f1_score": f1_score(golds, preds, zero_division=0),
            "auc": roc_auc_score(golds, probs) if len(set(golds)) > 1 else float("nan"),
            "precision": precision_score(golds, preds, zero_division=0),
            "recall": recall_score(golds, preds, zero_division=0),
            "true_positives": int(tp), "true_negatives": int(tn),
            "false_positives": int(fp), "false_negatives": int(fn)}


def run_on_dfs(train, test, data_column, label_column,
               max_seq_length=128,
               batch_size=32,
               learning_rate=2e-5,
               num_train_epochs=3,
               warmup_proportion=0.1,
               save_summary_steps=100,
               bert_model="bert-base-cased",
               output_dir="output"):
    tokenizer = AutoTokenizer.from_pretrained(bert_model)
    model = AutoModelForSequenceClassification.from_pretrained(bert_model, num_labels=2).to(device)
    print("Using BERT from %s (vocab size=%d)" % (bert_model, tokenizer.vocab_size))

    train_loader = DataLoader(make_features(train, tokenizer, max_seq_length, data_column, label_column),
                              batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(make_features(test, tokenizer, max_seq_length, data_column, label_column),
                             batch_size=batch_size)

    num_train_steps = len(train_loader) * num_train_epochs
    num_warmup_steps = int(num_train_steps * warmup_proportion)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_train_steps)
    writer = SummaryWriter(output_dir)          # Tensorboard 기록

    results, step = [], 0
    for epoch in range(num_train_epochs):
        model.train()
        for input_ids, attn, seg, labels in train_loader:
            out = model(input_ids=input_ids.to(device), attention_mask=attn.to(device),
                        token_type_ids=seg.to(device), labels=labels.to(device))
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
            step += 1
            if step % save_summary_steps == 0:
                writer.add_scalar("train/loss", out.loss.item(), step)

        print("End of epoch %d." % (epoch + 1))
        result_dict = evaluate(model, test_loader)
        for k in ["loss", "eval_accuracy", "f1_score", "auc"]:
            writer.add_scalar(f"eval/{k}", result_dict[k], epoch + 1)
        print(result_dict)
        results.append(result_dict)

    writer.close()
    return results, model


def pretty_print(result):
    df = pd.DataFrame([result]).T
    df.columns = ["values"]
    return df


In [ ]:
def load_data(data_file):
    data = pd.read_csv(data_file)

    # Only use the top quartile as polite, and bottom quartile as impolite. Discard the rest.
    quantiles = data["Normalized Score"].quantile([0.25, 0.5, 0.75])
    # print(quantiles)

    for i in range(len(data)):
        score = data.loc[i, "Normalized Score"]
        if score <= quantiles[0.25]:
            # Bottom quartile (impolite).
            data.loc[i, "Normalized Score"] = 0
        elif score >= quantiles[0.75]:
            # Top quartile (polite).
            data.loc[i, "Normalized Score"] = 1
        else:
            # Neutral.
            data.loc[i, "Normalized Score"] = 2

    data["Normalized Score"] = data["Normalized Score"].astype(int)

    # Discard neutral examples.
    data = data[data["Normalized Score"] < 2]
    
    data = data.sample(frac=1, random_state=42).reset_index(drop=True)   # [수정] 섞은 결과를 저장하지 않던 버그 수정
    n_test = len(data) // 10
    test_data = data[:n_test]
    train_data = data[n_test:]
    
    print("Data loaded successfully. Train=%d, test=%d, total=%d." % (len(train_data), len(test_data), len(train_data) + len(test_data)))
    print("Some train samples:")
    print(train_data.head())
    print("Some test samples:")
    print(test_data.head())

    return train_data, test_data

In [ ]:
train_data, test_data = load_data("Stanford_politeness_corpus/wikipedia.annotated.csv")

params = {
    "data_column": "Request",
    "label_column": "Normalized Score",
    "batch_size": 16,
    "num_train_epochs": 3,
    "bert_model": "bert-base-cased"     # 원본의 tfhub bert_cased_L-12_H-768_A-12와 같은 모델
}

result, model = run_on_dfs(train_data, test_data, **params)
pretty_print(result[-1])


## Tensorboard를 이용한 학습 결과 시각화

[Tensorboard](https://www.tensorflow.org/tensorboard)는 TensorFlow와 함께 제공되는 시각화 도구입니다.
단순히 학습 중 loss 함수의 값, 성능 뿐만이 아니라 사진, 글, 임베딩 등 다양한 정보들을 기록하고, 확인할 수 있습니다.

앞서 실습한 코드는 PyTorch의 `SummaryWriter`로 학습 손실(train/loss)과 에폭별 평가 지표(eval/…)를 기록하도록 되어 있습니다. 좌측 파일 목록을 보시면 "output"이라는 폴더가 생겼고, 그 아래 "events.out.tfevents"로 시작하는 파일이 생성된 것을 보실 수 있습니다.

Colab에서 Tensorboard를 확인하려면 아래 코드를 실행하면 됩니다.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir output